In [ ]:
!pip install optimum[onnxruntime] transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
MODEL_PATH = "/content/drive/MyDrive/notification_model/final_model"
ONNX_OUTPUT_PATH = "/content/drive/MyDrive/notification_model/distilbert_onnx"


In [ ]:
import os
os.makedirs(ONNX_OUTPUT_PATH, exist_ok=True)


In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification

model = ORTModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    export=True
)

model.save_pretrained(ONNX_OUTPUT_PATH)

print("ONNX model exported successfully.")


Multiple distributions found for package optimum. Picked distribution: optimum-onnx
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


ONNX model exported successfully.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.save_pretrained(ONNX_OUTPUT_PATH)

print("Tokenizer saved successfully.")


Tokenizer saved successfully.


In [ ]:
import json
import os

# Because you trained on priority values
label_list = [1, 2, 3, 4]

label_map = {i: label for i, label in enumerate(label_list)}

with open(os.path.join(ONNX_OUTPUT_PATH, "label_map.json"), "w") as f:
    json.dump(label_map, f)

print("Label map saved successfully.")


Label map saved successfully.


In [ ]:
os.listdir(ONNX_OUTPUT_PATH)


['model.onnx',
 'config.json',
 'label_map.json',
 'tokenizer_config.json',
 'vocab.txt',
 'special_tokens_map.json',
 'tokenizer.json']

In [ ]:
import torch
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

model = ORTModelForSequenceClassification.from_pretrained(ONNX_OUTPUT_PATH)
tokenizer = AutoTokenizer.from_pretrained(ONNX_OUTPUT_PATH)

text = "Your order has been shipped."

inputs = tokenizer(text, return_tensors="pt")

outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1).item()

print("Predicted encoded label:", prediction)


Predicted encoded label: 3


In [ ]:
import torch
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

model = ORTModelForSequenceClassification.from_pretrained(ONNX_OUTPUT_PATH)
tokenizer = AutoTokenizer.from_pretrained(ONNX_OUTPUT_PATH)

text = "Congratulations! You won a free prize!"

inputs = tokenizer(text, return_tensors="pt")

outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1).item()

print("Predicted encoded label:", prediction)


Predicted encoded label: 0


**QUANTIZATION**

In [ ]:
!pip install onnxruntime onnxruntime-tools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.7/212.7 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 12.3 MB/s eta 0:00:00


In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

In [ ]:
import os

original_model_path = os.path.join(ONNX_OUTPUT_PATH, "model.onnx")
quantized_model_path = os.path.join(ONNX_OUTPUT_PATH, "model_quantized.onnx")

In [ ]:
quantize_dynamic(
    model_input=original_model_path,
    model_output=quantized_model_path,
    weight_type=QuantType.QInt8
)

print("Quantization complete!")

Quantization complete!


In [ ]:
import os

original_size = os.path.getsize(original_model_path) / (1024 * 1024)
quantized_size = os.path.getsize(quantized_model_path) / (1024 * 1024)

print(f"Original model size: {original_size:.2f} MB")
print(f"Quantized model size: {quantized_size:.2f} MB")

Original model size: 255.53 MB
Quantized model size: 64.23 MB


In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer
import torch

model = ORTModelForSequenceClassification.from_pretrained(
    ONNX_OUTPUT_PATH,
    file_name="model_quantized.onnx"
)

tokenizer = AutoTokenizer.from_pretrained(ONNX_OUTPUT_PATH)

text = "Your order has been shipped."
inputs = tokenizer(text, return_tensors="pt")

outputs = model(**inputs)
prediction = torch.argmax(outputs.logits, dim=1).item()

print("Prediction from quantized model:", prediction)

Prediction from quantized model: 2
